# Contador de itens em esteira — subtração de background

Avaliação I — Visão Computacional (prof. Lúcio Campos)

## Etapa 1 · parte 1 — leitura e inspeção dos vídeos

Esta parte cobre a leitura dos dois vídeos com OpenCV, os metadados e a inspeção de frames. Subtração de background (MOG2/KNN), limiarização, morfologia, contornos e contagem pela linha virtual entram nas próximas partes da Etapa 1; a avaliação é a Etapa 2.

**Colab:** envie `01.mp4` e `test_video.mp4` pelo painel de arquivos (vão para `/content`). O armazenamento do Colab some ao reiniciar o ambiente; para algo durável, monte o Drive e aponte `VIDEO_DIR` para a pasta com os vídeos.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("OpenCV", cv2.__version__)

try:
    import google.colab  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

# Colab: o painel de arquivos envia para /content (com o Drive montado: Path("/content/drive/MyDrive/..."))
VIDEO_DIR = Path("/content") if EM_COLAB else Path("../data/raw")
VIDEOS = {"01": VIDEO_DIR / "01.mp4", "test_video": VIDEO_DIR / "test_video.mp4"}

## 1. Funções de leitura

`cv2.VideoCapture` não levanta exceção quando o arquivo não abre: ele só devolve `isOpened() == False`. Por isso `abrir_video` checa explicitamente.

In [ ]:
def abrir_video(caminho):
    """Abre o vídeo ou falha com mensagem clara."""
    cap = cv2.VideoCapture(str(caminho))
    if not cap.isOpened():
        raise FileNotFoundError(f"Não foi possível abrir {caminho}")
    return cap


def metadados(caminho):
    """Resolução, fps e nº de frames declarado pelo container."""
    cap = abrir_video(caminho)
    meta = {
        "largura": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "altura": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "frames_declarados": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()
    meta["duracao_s"] = meta["frames_declarados"] / meta["fps"] if meta["fps"] else float("nan")
    return meta


def contar_frames_lidos(caminho):
    """Decodifica o vídeo inteiro e conta os frames realmente lidos."""
    cap = abrir_video(caminho)
    n = 0
    while cap.read()[0]:
        n += 1
    cap.release()
    return n


def ler_frame(caminho, indice):
    """Frame `indice` do vídeo (BGR, uint8)."""
    cap = abrir_video(caminho)
    cap.set(cv2.CAP_PROP_POS_FRAMES, indice)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise ValueError(f"Frame {indice} não pôde ser lido de {caminho}")
    return frame


def mostrar(frames, titulos):
    """Exibe frames BGR lado a lado (cv2.imshow não funciona no Colab)."""
    fig, eixos = plt.subplots(1, len(frames), figsize=(5 * len(frames), 4))
    for ax, f, t in zip(np.atleast_1d(eixos), frames, titulos):
        ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))  # OpenCV é BGR, matplotlib é RGB
        ax.set_title(t)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 2. Metadados

Compara o nº de frames **declarado** pelo container (`CAP_PROP_FRAME_COUNT`) com o nº de frames **realmente decodificados**.

In [ ]:
META = {}
for nome, caminho in VIDEOS.items():
    m = metadados(caminho)
    m["frames_lidos"] = contar_frames_lidos(caminho)
    META[nome] = m
    print(f"{nome:>10}: {m['largura']}x{m['altura']} | {m['fps']:.2f} fps | "
          f"declarados={m['frames_declarados']} lidos={m['frames_lidos']} | {m['duracao_s']:.1f} s")

## 3. Inspeção de frames

Três frames por vídeo (5%, 40% e 75% da duração) para ver a cena, a iluminação e o sentido do movimento.

In [ ]:
for nome, caminho in VIDEOS.items():
    n = META[nome]["frames_lidos"]
    indices = [int(n * f) for f in (0.05, 0.40, 0.75)]
    mostrar([ler_frame(caminho, i) for i in indices], [f"{nome} — frame {i}" for i in indices])

## 4. Observações da etapa 1

*(preencher com as suas observações)*

- **Cena e câmera:** a câmera é fixa? Há perspectiva (itens mudam de tamanho)?
- **Sobreposições na imagem:** timestamp, marca d'água ou outros elementos que mudam e podem virar foreground.
- **Iluminação:** estável ou variável ao longo do vídeo?
- **Itens:** aparência, tamanho aproximado em pixels e sentido do deslocamento. Isso define a posição e a orientação da linha virtual.
- **Contagem manual (etapa 0):** regra adotada e valor para cada vídeo.

## Etapa 1 · parte 2 — subtração de background e máscara limpa

**Ideia.** Cada pixel tem um modelo do que é "fundo". O **MOG2** modela a intensidade de cada pixel como uma mistura de gaussianas e marca como foreground o pixel que não se encaixa em nenhuma componente de fundo. O **KNN** compara o pixel com amostras recentes e marca como foreground se há poucos vizinhos próximos. Nos dois, o modelo se adapta com o tempo.

`apply()` devolve uma máscara com 3 valores: **0** fundo, **127** sombra (com `detectShadows=True`) e **255** foreground. Limiarizar em ~200 descarta as sombras. Depois, **abertura** (erosão + dilatação) remove ruído pequeno e **fechamento** (dilatação + erosão) fecha buracos dentro dos objetos.

**Pré-condições e limitações.** Câmera fixa. O modelo precisa de alguns frames para estabilizar (*warm-up*). Objeto parado vira fundo depois de um tempo, e objetos encostados formam um único blob.

In [ ]:
# Parâmetros por vídeo. `ignorar`: regiões (x0, y0, x1, y1) em fração da imagem, p/ timestamp e marcas d'água.
PARAMS = {
    "01": dict(escala=1.0, blur=5, k_abrir=3, k_fechar=9,
               ignorar=[(0.0, 0.0, 0.70, 0.12), (0.70, 0.84, 1.0, 1.0)]),
    "test_video": dict(escala=0.5, blur=5, k_abrir=5, k_fechar=11,
                       ignorar=[(0.93, 0.92, 1.0, 1.0)]),
}
LIMIAR_FG = 200  # > 127 (sombra) e <= 255 (foreground)


def novo_subtrator(tipo):
    if tipo == "MOG2":
        return cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=25, detectShadows=True)
    return cv2.createBackgroundSubtractorKNN(history=500, dist2Threshold=400.0, detectShadows=True)


def mascara_roi(forma, ignorar):
    """255 onde vale, 0 nas regiões a ignorar."""
    h, w = forma[:2]
    roi = np.full((h, w), 255, np.uint8)
    for x0, y0, x1, y1 in ignorar:
        roi[int(y0 * h):int(y1 * h), int(x0 * w):int(x1 * w)] = 0
    return roi


def limpar(bruta, k_abrir, k_fechar):
    """Limiariza (descarta sombra) e aplica abertura + fechamento."""
    binaria = cv2.threshold(bruta, LIMIAR_FG, 255, cv2.THRESH_BINARY)[1]
    abrir = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_abrir, k_abrir))
    fechar = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_fechar, k_fechar))
    binaria = cv2.morphologyEx(binaria, cv2.MORPH_OPEN, abrir)
    return cv2.morphologyEx(binaria, cv2.MORPH_CLOSE, fechar)


def percorrer(caminho, tipo, p, amostras=()):
    """Passa o vídeo inteiro pelo subtrator.

    Retorna a fração de foreground (máscara bruta == 255) por frame e
    {indice: (frame redimensionado, máscara bruta)} para os índices em `amostras`.
    """
    sub = novo_subtrator(tipo)
    cap = abrir_video(caminho)
    roi, razao, guardados, i = None, [], {}, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if p["escala"] != 1.0:
            frame = cv2.resize(frame, None, fx=p["escala"], fy=p["escala"], interpolation=cv2.INTER_AREA)
        if roi is None:
            roi = mascara_roi(frame.shape, p["ignorar"])
        bruta = cv2.bitwise_and(sub.apply(cv2.GaussianBlur(frame, (p["blur"], p["blur"]), 0)), roi)
        razao.append(np.count_nonzero(bruta == 255) / np.count_nonzero(roi))
        if i in amostras:
            guardados[i] = (frame, bruta)
        i += 1
    cap.release()
    return np.array(razao), guardados


def mostrar_mascaras(nome, res, p):
    """Linhas = amostras; colunas = frame, MOG2 bruta, MOG2 limpa, KNN bruta, KNN limpa."""
    indices = sorted(res["MOG2"][1])
    fig, eixos = plt.subplots(len(indices), 5, figsize=(18, 3.3 * len(indices)), squeeze=False)
    for linha, i in zip(eixos, indices):
        frame = res["MOG2"][1][i][0]
        linha[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        linha[0].set_title(f"{nome} — frame {i}")
        for col, tipo in ((1, "MOG2"), (3, "KNN")):
            bruta = res[tipo][1][i][1]
            linha[col].imshow(bruta, cmap="gray", vmin=0, vmax=255)
            linha[col].set_title(f"{tipo} bruta (127 = sombra)")
            linha[col + 1].imshow(limpar(bruta, p["k_abrir"], p["k_fechar"]), cmap="gray", vmin=0, vmax=255)
            linha[col + 1].set_title(f"{tipo} limpa")
    for ax in eixos.ravel():
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 5. MOG2 vs KNN lado a lado

Roda os dois subtratores no vídeo inteiro e mostra a máscara bruta e a limpa em alguns frames.

In [ ]:
AMOSTRAS = {"01": [300, 851, 1500], "test_video": [120, 360, 600]}

RES = {}
for nome, caminho in VIDEOS.items():
    RES[nome] = {t: percorrer(caminho, t, PARAMS[nome], AMOSTRAS[nome]) for t in ("MOG2", "KNN")}
    mostrar_mascaras(nome, RES[nome], PARAMS[nome])

## 6. Fração de foreground ao longo do tempo

Serve para ver o *warm-up* do modelo, os momentos de atividade e variações bruscas (iluminação, ruído).

In [ ]:
AQUECIMENTO = 5  # no 1º frame o modelo está vazio e tudo vira foreground (estoura a escala do gráfico)

fig, eixos = plt.subplots(len(VIDEOS), 1, figsize=(12, 3.5 * len(VIDEOS)))
for ax, (nome, caminho) in zip(np.atleast_1d(eixos), VIDEOS.items()):
    t = np.arange(META[nome]["frames_lidos"]) / META[nome]["fps"]
    for tipo, (razao, _) in RES[nome].items():
        ax.plot(t[AQUECIMENTO:], 100 * razao[AQUECIMENTO:], label=tipo, lw=1)
    ax.set_title(nome)
    ax.set_xlabel("tempo (s)")
    ax.set_ylabel("foreground (% da ROI)")
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Observações da parte 2

*(preencher com as suas observações)*

- **MOG2 vs KNN:** qual separou melhor o objeto do fundo em cada vídeo? Qual deixou mais ruído e mais sombra (cinza = 127 na máscara bruta)?
- **Efeito da morfologia:** o que a abertura removeu e o que o fechamento preencheu? Algum objeto foi fundido com outro ou partido em dois?
- **Gráfico de foreground:** o que explica o pico no início e as quedas bruscas? Há algo periódico? (confira o tempo exato das quedas no vídeo de maçãs)
- **Falhas visíveis até aqui:** estruturas estáticas aparecendo como foreground, reflexos, itens encostados. Essas falhas alimentam a análise da Etapa 2.